In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from langchain_huggingface import HuggingFacePipeline, ChatHuggingFace
from langchain.agents import create_agent

def get_weather(city: str) -> str:
    """Get weather for a given city."""
    return f"It's always sunny in {city}!"

MODEL_ID = "Qwen/Qwen2.5-7B-Instruct" # Tool calling対応のモデルでないとget_weatherが呼べないので注意

# Create HuggingFace pipeline for text generation
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype="auto",
    device_map="auto"
)
pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=1024,
    temperature=0.0,    # Tool callingしやすくする
    do_sample=False,    # Tool callingしやすくする
    return_full_text=False       # プロンプト自体をレスポンスに含めない（LangChainで必須）
)
hf_pipeline = HuggingFacePipeline(pipeline=pipe)
chat_model = ChatHuggingFace(llm=hf_pipeline)

agent = create_agent(
    model=chat_model,
    tools=[get_weather],
    system_prompt="You are a helpful assistant",
)

result = agent.invoke(
    {"messages": [{"role": "user", "content": "What's the weather in San Francisco?"}]}
)
print(result["messages"][-1].content_blocks)